# Detroit 2024: Datenqualität Schritt für Schritt

Dieses Notebook prüft die Bronze-Schicht des Jahrgangs 2024 — Schritt für
Schritt, jeder mit einer Frage, einer Prüfung und einem Befund. Es ist das
Gegenstück zu Notebook 03 für Chicago.

**Wichtig für die Einordnung:** Bronze ist unverändert aus der ArcGIS-API
übernommen. Es wird nichts geparst, nichts korrigiert, nichts weggeworfen. Was
hier auffällt, ist deshalb eine Eigenschaft **der Quelle**, nicht ein Fehler
der Ingestion. Das Bereinigen gehört nach Silver.

Detroit stellt dabei zwei Fragen, die Chicago nicht stellt:

* Die Quelle liefert **eine Zeile je Anklage**, nicht je Delikt. Wer beide
  Städte summiert, ohne das zu wissen, zählt Detroit zu hoch.
* Es liegen **zwei Snapshots desselben Jahrgangs** nebeneinander, gut 20
  Stunden auseinander. Damit lässt sich hier etwas messen, was bei Chicago
  offenbleibt: wie stabil ein abgeschlossenes Jahr überhaupt ist.

Am Ende steht eine Übersicht aller Prüfungen mit Ampelstatus.

| | |
| --- | --- |
| Datenbestand | `data/bronze/city=detroit/`, alle Läufe des Jahrgangs 2024 |
| Grundlage | ArcGIS FeatureServer `RMS_Crime_Incidents`, Schema aus dem Manifest |
| Referenz | `config/micr_nibrs_mapping.csv` |

## Schritt 1 – Daten laden

Gelesen wird über DuckDB: Detroits Bronze liegt als Parquet mit einer
verschachtelten Spalte `attributes`, und ein View darüber spart das Auspacken
in Python.

Welche Läufe gelten, entscheidet **das Manifest**, nicht das Verzeichnis. Nur
was `status = SUCCESS` trägt, wird gelesen — genau so macht es auch
`silver_pipeline.build_crime_offense`.

`BEFUNDE` sammelt unterwegs das Ergebnis jeder Prüfung ein — als Dictionary,
damit eine erneut ausgeführte Zelle den Eintrag ersetzt statt ihn zu verdoppeln.

In [ ]:
import json
from pathlib import Path

import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src' / 'crime' / 'detroit_pipeline').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'crime' / 'detroit_pipeline').exists():
    raise RuntimeError('Projekt-Root mit src/crime/detroit_pipeline nicht gefunden')

JAHR = 2024
ZONE = 'America/Detroit'
DATA_ROOT = PROJECT_ROOT / 'data'
BRONZE = DATA_ROOT / 'bronze' / 'city=detroit'
BEFUNDE: dict[str, tuple[str, str]] = {}

laeufe = []
for pfad in sorted(BRONZE.glob('**/run_*/manifest.json')):
    if any(teil.startswith('.') for teil in pfad.parts):
        continue  # .partial: noch nicht veroeffentlicht
    m = json.loads(pfad.read_text(encoding='utf-8'))
    if str(m.get('status', '')).upper() != 'SUCCESS':
        continue
    laeufe.append((pfad.parent / m.get('bronze_file', 'part-0001.parquet'), m))

if not laeufe:
    raise RuntimeError(
        'Keine erfolgreichen Bronze-Laeufe gefunden. Erst laden mit:\n'
        '  python -m detroit_pipeline.extract_detroit --mode delta --years 2024'
    )

laeufe.sort(key=lambda t: t[1]['finished_at'])
neuster, manifest = laeufe[-1]

con = duckdb.connect()
con.execute("CREATE VIEW d AS SELECT attributes.* FROM read_parquet('" + str(neuster) + "')")
con.execute("CREATE VIEW roh AS SELECT * FROM read_parquet('" + str(neuster) + "')")

N = con.execute('SELECT count(*) FROM d').fetchone()[0]
felder = [(r[0], r[1]) for r in con.execute('DESCRIBE SELECT * FROM d').fetchall()]

print('Erfolgreiche Laeufe :', len(laeufe))
for _, m in laeufe:
    print('  ', m['pipeline_run_id'], m['finished_at'][:19], f"{m['downloaded_count']:,} Zeilen")
print('Ausgewertet wird    :', manifest['pipeline_run_id'])
print('Zeilen              :', f'{N:,}')
print('Felder              :', len(felder))

## Schritt 2 – Was das Manifest bereits beweist

Bevor wir die Daten selbst anfassen: Einiges ist schon während der Ingestion
geprüft worden und muss hier nicht wiederholt werden. Das Manifest hält es fest.

Drei Zusagen stehen darin:

* **`snapshot_membership_verified`** — die Ingestion hat sich vom Server die
  Liste aller ObjectIDs geben lassen und gegen das Geladene abgeglichen. Nicht
  „gleich viele", sondern *dieselben*.
* **`source_count_before` / `source_count_after`** — die Zeilenzahl wurde vor
  und nach dem Lauf abgefragt. Sind sie gleich, hat sich die Quelle während des
  Ladens nicht bewegt; die 46 Seiten gehören zu einem Stand.
* **`bronze_sha256`** und die Raw-Schicht — jede ArcGIS-Antwort liegt zusätzlich
  unverändert als gzip daneben, mit eigener Prüfsumme. Was hier gemessen wird,
  ist damit auf die gelieferten Bytes zurückführbar.

Eine Zusage fehlt bewusst: `captures_source_deletions = False`. Ein voller
Snapshot sieht nur, was da ist. Was seit dem letzten Lauf *gelöscht* wurde,
merkt erst der Vergleich zweier Snapshots — Schritt 3.

In [ ]:
zusagen = {
    'snapshot_membership_verified': manifest.get('snapshot_membership_verified'),
    'source_count_before': manifest.get('source_count_before'),
    'source_count_after': manifest.get('source_count_after'),
    'downloaded_count': manifest.get('downloaded_count'),
    'snapshot_object_id_count': manifest.get('snapshot_object_id_count'),
    'source_changed_during_run': manifest.get('source_changed_during_run'),
    'captures_source_deletions': manifest.get('captures_source_deletions'),
    'rows_with_extra': manifest.get('rows_with_extra'),
}
display(pd.DataFrame(zusagen.items(), columns=['Zusage', 'Wert']))

vollstaendig = (
    manifest.get('snapshot_membership_verified') is True
    and manifest.get('source_count_before') == manifest.get('source_count_after')
    == manifest.get('downloaded_count') == N
)
BEFUNDE['Snapshot nachgewiesen'] = (
    ('OK', f'{N:,} Zeilen, ObjectID-Menge abgeglichen, Quelle waehrend des Laufs stabil')
    if vollstaendig else
    ('ACHTUNG', 'Manifest und Datei sind sich nicht einig')
)

raw = manifest.get('raw_layer')
if raw:
    print('Raw-Schicht :', raw['path'])
    print('Antworten   :', raw.get('responses'))
    BEFUNDE['Raw-Schicht vorhanden'] = ('OK', str(raw.get('responses')) + ' Antworten unveraendert abgelegt')
else:
    BEFUNDE['Raw-Schicht vorhanden'] = ('HINWEIS', 'Dieser Lauf stammt aus der Zeit vor der Raw-Schicht')

## Schritt 3 – Zwei Snapshots desselben Jahrgangs

2024 ist seit acht Monaten abgeschlossen. Naheliegende Annahme: Die Zahlen
stehen fest. Zwei Läufe im Abstand von gut 20 Stunden erlauben, das
nachzurechnen — und die Annahme hält nicht.

Zwei Dinge werden getrennt geprüft:

1. **Welches Feld identifiziert eine Zeile über Läufe hinweg?** `ESRI_OID` ist
   der ObjectID des FeatureServers und wird für die Seitennavigation benutzt.
   Die Frage ist, ob er auch eine *Identität* ist.
2. **Was hat sich inhaltlich geändert?** Join über den stabilen Schlüssel,
   Feld für Feld.

Der Schritt wird übersprungen, wenn nur ein Lauf vorliegt.

In [ ]:
if len(laeufe) < 2:
    print('Nur ein Lauf vorhanden - Schritt 3 uebersprungen.')
    BEFUNDE['Quelle stabil ueber Laeufe'] = ('HINWEIS', 'nur ein Snapshot vorhanden, nicht pruefbar')
else:
    vorheriger = laeufe[-2][0]
    con.execute("CREATE OR REPLACE VIEW v AS SELECT attributes.* FROM read_parquet('"
                + str(vorheriger) + "')")
    abstand = pd.Timestamp(manifest['finished_at']) - pd.Timestamp(laeufe[-2][1]['finished_at'])

    schluessel = []
    for feld in ('ESRI_OID', 'incident_entry_id', 'case_id'):
        nur_alt = con.execute(
            'SELECT count(*) FROM (SELECT ' + feld + ' FROM v EXCEPT SELECT ' + feld + ' FROM d)'
        ).fetchone()[0]
        nur_neu = con.execute(
            'SELECT count(*) FROM (SELECT ' + feld + ' FROM d EXCEPT SELECT ' + feld + ' FROM v)'
        ).fetchone()[0]
        schluessel.append({'Feld': feld, 'nur im alten Lauf': nur_alt, 'nur im neuen Lauf': nur_neu})
    print('Abstand der beiden Laeufe:', abstand)
    display(pd.DataFrame(schluessel))

`ESRI_OID` fällt durch: Fast jede Zeile trägt im zweiten Lauf eine andere. Der
ObjectID wird bei jeder Abfrage neu vergeben — er nummeriert das *Ergebnis*,
nicht den Datensatz. Brauchbar ist er nur innerhalb eines Laufs, für die
Seitennavigation und für `snapshot_ids_sha256`. **Zwei `snapshot_ids_sha256`
verschiedener Läufe zu vergleichen, sagt nichts aus.**

`incident_entry_id` und `case_id` halten dagegen ohne Abweichung. Damit lässt
sich vergleichen, was sich am Inhalt geändert hat.

In [ ]:
if len(laeufe) >= 2:
    zeilen = []
    for feld, _typ in felder:
        if feld == 'ESRI_OID':
            continue
        anz = con.execute(
            'SELECT count(*) FROM v JOIN d USING (incident_entry_id) '
            'WHERE v.' + feld + ' IS DISTINCT FROM d.' + feld
        ).fetchone()[0]
        if anz:
            zeilen.append({'Feld': feld, 'geaenderte Zeilen': anz, 'Anteil': f'{anz / N:.3%}'})
    display(pd.DataFrame(zeilen).sort_values('geaenderte Zeilen', ascending=False))

    # Nur Polsterung oder wirklich ein anderer Wert?
    roh_geaendert = con.execute(
        'SELECT count(*) FROM v JOIN d USING (incident_entry_id) '
        'WHERE v.offense_description IS DISTINCT FROM d.offense_description'
    ).fetchone()[0]
    echt = con.execute(
        'SELECT count(*) FROM v JOIN d USING (incident_entry_id) '
        'WHERE trim(v.offense_description) IS DISTINCT FROM trim(d.offense_description)'
    ).fetchone()[0]
    print('offense_description roh geaendert :', roh_geaendert)
    print('davon wirklich anderer Wert       :', echt,
          '  (Rest: nur Leerzeichen, siehe Schritt 8)')

In [ ]:
if len(laeufe) >= 2:
    display(con.execute(
        'SELECT v.incident_entry_id AS id, '
        '       trim(v.offense_description) AS vorher, '
        '       trim(d.offense_description) AS nachher, '
        '       epoch_ms(d.updated_at)      AS updated_at_neu '
        'FROM v JOIN d USING (incident_entry_id) '
        'WHERE trim(v.offense_description) IS DISTINCT FROM trim(d.offense_description) '
        'ORDER BY id'
    ).df())

    ohne_stempel = con.execute(
        'SELECT count(*) FROM v JOIN d USING (incident_entry_id) '
        'WHERE trim(v.offense_description) IS DISTINCT FROM trim(d.offense_description) '
        '  AND v.updated_at IS NOT DISTINCT FROM d.updated_at'
    ).fetchone()[0]
    BEFUNDE['Quelle stabil ueber Laeufe'] = ('ACHTUNG', (
        f'{echt} Delikte in {abstand.components.hours} Stunden umklassiert, '
        f'{ohne_stempel} davon ohne neues updated_at'
    ))
    BEFUNDE['ESRI_OID als Schluessel'] = (
        'ACHTUNG', 'nicht stabil ueber Laeufe - nur laufintern brauchbar')

Zehn Delikte sind in gut 20 Stunden umklassiert worden — eine tätliche
Auseinandersetzung wird zu einer Sachbeschädigung, eine schwere Körperverletzung
zu einer Schussabgabe ohne Todesfolge. Das sind 0,01 % und verschiebt kein
Ergebnis. Der Punkt ist ein anderer: **Ein abgeschlossenes Jahr ist keine
Konstante.** Jede Zahl in diesem Projekt gilt für einen Stand, und der Stand
muss dranstehen.

Beruhigend ist der zweite Teil: **Jede inhaltliche Änderung trägt einen neuen
`updated_at`.** Damit funktioniert die Deduplizierung in Silver — sie nimmt je
`incident_entry_id` die Fassung mit dem jüngsten `updated_at` und trifft
nachweislich die richtige.

## Schritt 4 – Welcher Schlüssel trägt

Vier Felder sehen nach einem Schlüssel aus. Nur eines ist einer.

In [ ]:
kandidaten = []
for feld in ('incident_entry_id', 'case_id', 'crime_id', 'report_number'):
    verschieden = con.execute('SELECT count(DISTINCT trim(' + feld + ')) FROM d').fetchone()[0]
    leer = con.execute(
        'SELECT count(*) FROM d WHERE ' + feld + " IS NULL OR trim(" + feld + ") = ''"
    ).fetchone()[0]
    kandidaten.append({
        'Feld': feld,
        'verschiedene Werte': verschieden,
        'ohne Wert': leer,
        'eindeutig': 'ja' if verschieden == N and leer == 0 else 'nein',
    })
display(pd.DataFrame(kandidaten))
BEFUNDE['Schluessel'] = (
    'OK', f'incident_entry_id eindeutig ueber {N:,} Zeilen; case_id ist der Vorfall')

`incident_entry_id` ist eindeutig — und setzt sich sichtbar aus Vorfall und
MICR-Code zusammen (`1164528-3072`). `case_id`, `crime_id` und `report_number`
bezeichnen den **Vorfall** und wiederholen sich deshalb. Das ist kein Mangel,
sondern der Hinweis auf das eigentliche Thema: die Granularität.

## Schritt 5 – Die Granularität: eine Zeile ist eine Anklage

Chicago liefert eine Zeile je Delikt. Detroit liefert eine Zeile je **Anklage**.
Ein Überfall mit Schüssen und geraubtem Auto steht bei Detroit als mehrere
Zeilen, bei Chicago als eine.

Welches Feld die *Tat* trägt und welches die *Anklage*, sagen die Daten selbst:
Bei Vorfällen mit mehreren Zeilen bleibt `offense_description` deutlich öfter
konstant als `charge_description`. Die Tat ist also `offense_description`.

In [ ]:
tat_konstant, anklage_konstant, mehrzeilig = con.execute(
    'SELECT count(*) FILTER (WHERE beschreibungen = 1), '
    '       count(*) FILTER (WHERE anklagen = 1), '
    '       count(*) '
    'FROM (SELECT trim(case_id), '
    '             count(DISTINCT trim(offense_description)) AS beschreibungen, '
    '             count(DISTINCT trim(charge_description))  AS anklagen '
    '      FROM d GROUP BY 1 HAVING count(*) > 1)'
).fetchone()
print('Vorfaelle mit mehr als einer Zeile :', f'{mehrzeilig:,}')
print('   offense_description konstant    :',
      f'{tat_konstant:,} ({tat_konstant / mehrzeilig:.1%})')
print('   charge_description konstant     :',
      f'{anklage_konstant:,} ({anklage_konstant / mehrzeilig:.1%})')

zeilen_, vorfaelle, taten = con.execute(
    'SELECT count(*), count(DISTINCT trim(case_id)), '
    '       count(DISTINCT (trim(case_id), trim(offense_description))) FROM d'
).fetchone()
display(pd.DataFrame([
    {'Ebene': 'Zeilen (Anklagen)', 'Anzahl': zeilen_},
    {'Ebene': 'Taten (Vorfall x Delikt)', 'Anzahl': taten},
    {'Ebene': 'Vorfaelle (case_id)', 'Anzahl': vorfaelle},
]))

abweichend = con.execute(
    'SELECT count(*) FROM d WHERE trim(charge_description) <> trim(offense_description)'
).fetchone()[0]
print()
print('Anklage weicht von der Tat ab :', f'{abweichend:,} ({abweichend / N:.1%})')
BEFUNDE['Granularität'] = ('ACHTUNG', (
    f'{zeilen_:,} Anklagen zu {taten:,} Taten - wer Zeilen zaehlt, '
    f'zaehlt Detroit um {zeilen_ / taten - 1:.1%} zu hoch'
))

Der Unterschied ist keine Feinheit: **Zeilen zu zählen liefert 3,9 % zu viel.**
Silver behält deshalb jede Anklage — sie ist eine echte Information — markiert
aber je Tat genau **eine** zählende Zeile (`counts`), und Gold nimmt nur diese.

Die 3,6 % der Zeilen, bei denen Anklage und Tat auseinandergehen, sind der Fall,
auf den Artan gestossen ist: Ein Vorfall mit `offense_description = LARCENY` kann
eine Anklage `CSC 3RD DEGREE` tragen. Die Tat war ein Diebstahl, die Anklage
lautet auf sexuelle Nötigung — beides stimmt, es sind zwei verschiedene Fragen.

## Schritt 6 – Fehlende Werte je Feld

Gezählt wird getrennt nach `NULL` und leerer Zeichenkette. Beides bedeutet
„kein Wert", aber nur eines davon fällt beim Lesen auf.

In [ ]:
zeilen = []
for feld, typ in felder:
    if typ.startswith('VARCHAR'):
        nul, leer = con.execute(
            'SELECT count(*) FILTER (WHERE ' + feld + ' IS NULL), '
            '       count(*) FILTER (WHERE ' + feld + ' IS NOT NULL AND trim('
            + feld + ") = '') FROM d"
        ).fetchone()
    else:
        nul = con.execute('SELECT count(*) FROM d WHERE ' + feld + ' IS NULL').fetchone()[0]
        leer = 0
    if nul or leer:
        zeilen.append({'Feld': feld, 'Typ': typ, 'NULL': nul, 'leer': leer,
                       'Anteil': f'{(nul + leer) / N:.2%}'})
luecken = pd.DataFrame(zeilen).sort_values('NULL', ascending=False)
display(luecken)
print('Felder ohne jede Luecke:', len(felder) - len(luecken), 'von', len(felder))

`updated_in_ibr_at` fehlt bei einem Viertel der Zeilen — das ist der Zeitpunkt
der Meldung an das bundesweite IBR und sagt nichts über die Tat. Die übrigen
Lücken liegen unter 1,5 %, und die geografischen hängen alle am selben Befund:
Schritt 7.

Eine Lücke fällt aus dem Muster: 15 Zeilen tragen bei `offense_description` eine
**leere Zeichenkette** statt `NULL`. Wer nur auf `NULL` prüft, hält sie für
gefüllt — und gruppiert am Ende nach einem Delikttyp ohne Namen.

## Schritt 7 – Die Geo-Lücke

Der räumliche Join gegen die Census-Tracts trägt das ganze Projekt. Die Frage
ist nicht nur *wie viele* Koordinaten fehlen, sondern **ob sie zufällig
fehlen**. Bei Chicago tun sie das nicht — dort fehlen Sexualdelikte deutlich
überproportional, weil die Adresse geschützt wird. Für Detroit muss dieselbe
Frage gestellt werden.

Zusätzlich zwei Fallen, die eine reine Zählung nicht sieht: Koordinaten `0/0`
(die „Null-Insel" vor Afrika) und Punkte ausserhalb des Stadtrahmens.

In [ ]:
ohne_geo, null_insel = con.execute(
    'SELECT count(*) FILTER (WHERE latitude IS NULL OR longitude IS NULL), '
    '       count(*) FILTER (WHERE latitude = 0 AND longitude = 0) FROM d'
).fetchone()
rahmen = con.execute(
    'SELECT round(min(latitude), 4), round(max(latitude), 4), '
    '       round(min(longitude), 4), round(max(longitude), 4) '
    'FROM d WHERE latitude IS NOT NULL'
).fetchone()
print('ohne Koordinaten :', f'{ohne_geo:,} ({ohne_geo / N:.2%})')
print('Null-Insel (0/0) :', null_insel)
print('Breite           :', rahmen[0], '..', rahmen[1])
print('Laenge           :', rahmen[2], '..', rahmen[3])

verteilung = con.execute(
    'SELECT trim(offense_category) AS kategorie, count(*) AS zeilen, '
    '       count(*) FILTER (WHERE latitude IS NULL) AS ohne_geo, '
    '       round(100.0 * count(*) FILTER (WHERE latitude IS NULL) / count(*), 2) AS anteil_prozent '
    'FROM d GROUP BY 1 HAVING count(*) FILTER (WHERE latitude IS NULL) > 0 '
    'ORDER BY anteil_prozent DESC'
).df()
display(verteilung)

schnitt = ohne_geo / N * 100
auffaellig = verteilung[verteilung['anteil_prozent'] > 3 * schnitt]
BEFUNDE['Koordinaten vorhanden'] = ('OK' if ohne_geo / N < 0.01 else 'ACHTUNG', (
    f'{ohne_geo:,} Zeilen ohne Geo ({ohne_geo / N:.2%}), keine Null-Insel, '
    f'{len(auffaellig)} Kategorien deutlich ueber dem Schnitt'
))

0,29 % ohne Koordinaten, keine Null-Insel, alle Punkte im Stadtrahmen — das ist
weniger als bei Chicago (0,7 %).

Die Verteilung zeigt aber **dieselbe Schlagseite wie Chicago**, nur kleiner:
Sexualdelikte (1,84 %), Familiendelikte (2,27 %) und die Sammelkategorie `OTHER`
(2,58 %) liegen um das Sechs- bis Neunfache über dem Schnitt. Der Grund dürfte
derselbe sein — bei diesen Delikten wird die Adresse zurückgehalten.

Der Unterschied zu Chicago ist die Grössenordnung: Es geht um 16 von 869
Sexualdelikten. Die Richtung stimmt, das Gewicht nicht — bei einer Auswertung je
Delikttyp gehört die Abdeckung trotzdem mitangegeben.

Ein zweites Geo-Feld liefert Detroit gleich mit: `census_block_2020_geoid`. Es
erlaubt, den eigenen räumlichen Join gegen die Angabe der Stadt zu halten,
statt ihm blind zu vertrauen.

In [ ]:
bloecke, min_laenge, max_laenge, nicht_michigan = con.execute(
    'SELECT count(DISTINCT census_block_2020_geoid), '
    '       min(length(census_block_2020_geoid)), '
    '       max(length(census_block_2020_geoid)), '
    '       count(*) FILTER (WHERE census_block_2020_geoid IS NOT NULL '
    "                          AND left(census_block_2020_geoid, 2) <> '26') FROM d"
).fetchone()
print('verschiedene Bloecke :', f'{bloecke:,}')
print('Laenge               :', min_laenge, '..', max_laenge, 'Zeichen (GEOID Block = 15)')
print('ausserhalb Michigan  :', nicht_michigan)
BEFUNDE['Census-Block plausibel'] = (
    'OK', f'{bloecke:,} Bloecke, alle 15-stellig und in Michigan (FIPS 26)')

## Schritt 8 – Polsterung mit Leerzeichen

Die Textfelder kommen aus einem System mit fester Feldbreite. Ein Teil der Werte
ist auf diese Breite mit Leerzeichen aufgefüllt, ein anderer nicht — und zwar
**unvorhersehbar**: Derselbe Datensatz kam in Schritt 3 einmal mit und einmal
ohne Polsterung zurück.

Wer ungetrimmt gruppiert, bekommt denselben Delikttyp zweimal.

In [ ]:
zeilen = []
for feld in ('offense_description', 'charge_description', 'offense_category',
             'state_offense_code', 'case_status', 'neighborhood'):
    gepolstert, roh_werte, getrimmt = con.execute(
        'SELECT count(*) FILTER (WHERE ' + feld + ' <> trim(' + feld + ')), '
        '       count(DISTINCT ' + feld + '), count(DISTINCT trim(' + feld + ')) FROM d'
    ).fetchone()
    zeilen.append({
        'Feld': feld,
        'gepolsterte Zeilen': gepolstert,
        'Anteil': f'{gepolstert / N:.1%}',
        'Werte ungetrimmt': roh_werte,
        'Werte getrimmt': getrimmt,
    })
display(pd.DataFrame(zeilen))

schlimmster = max(zeilen, key=lambda z: z['Werte ungetrimmt'] - z['Werte getrimmt'])
BEFUNDE['Polsterung'] = ('ACHTUNG', (
    f"{schlimmster['Feld']}: {schlimmster['Werte ungetrimmt']} scheinbare Werte, "
    f"{schlimmster['Werte getrimmt']} echte - ohne trim() zaehlt jede Gruppierung doppelt"
))

Beide Beschreibungsfelder sind betroffen: `offense_description` hat ungetrimmt
152 verschiedene Werte, getrimmt 81 — `charge_description` 152 gegen 79. Rund
die Hälfte der scheinbaren Delikttypen wäre ein Artefakt der Feldbreite. Die
codierten Felder dagegen (`state_offense_code`, `offense_category`) sind sauber.

Silver trimmt deshalb jedes Textfeld beim Einlesen.

## Schritt 9 – Wertebereiche und welche Zeitzone gilt

`incident_occurred_at` sind Millisekunden seit Epoch. Damit daraus ein Zeitpunkt
wird, braucht es eine Zone — und die steht nirgends. Die Quelle liefert aber
vier abgeleitete Felder mit: `incident_time`, `incident_hour_of_day`,
`incident_day_of_week`, `incident_year`.

Diese vier lassen sich als Probe benutzen: Rechnen wir den Zeitstempel einmal
als UTC und einmal als Detroiter Ortszeit um — welche Lesart stimmt mit den
mitgelieferten Feldern überein?

In [ ]:
probe = con.execute(
    "WITH t AS (SELECT *, epoch_ms(incident_occurred_at) AS als_utc, "
    "                  timezone('" + ZONE + "', epoch_ms(incident_occurred_at) "
    "                  AT TIME ZONE 'UTC') AS als_ortszeit FROM d) "
    'SELECT count(*) FILTER (WHERE incident_hour_of_day IS DISTINCT FROM hour(als_utc)), '
    '       count(*) FILTER (WHERE incident_hour_of_day IS DISTINCT FROM hour(als_ortszeit)), '
    '       count(*) FILTER (WHERE incident_day_of_week IS DISTINCT FROM isodow(als_utc)), '
    '       count(*) FILTER (WHERE incident_day_of_week IS DISTINCT FROM isodow(als_ortszeit)), '
    "       count(*) FILTER (WHERE incident_time IS DISTINCT FROM strftime(als_utc, '%H:%M')), "
    "       count(*) FILTER (WHERE incident_time IS DISTINCT FROM strftime(als_ortszeit, '%H:%M')), "
    '       count(*) FILTER (WHERE incident_year IS DISTINCT FROM year(als_ortszeit)) '
    'FROM t'
).fetchone()
display(pd.DataFrame([
    {'Feld': 'incident_hour_of_day', 'Abweichung als UTC': probe[0], 'Abweichung als Ortszeit': probe[1]},
    {'Feld': 'incident_day_of_week', 'Abweichung als UTC': probe[2], 'Abweichung als Ortszeit': probe[3]},
    {'Feld': 'incident_time',        'Abweichung als UTC': probe[4], 'Abweichung als Ortszeit': probe[5]},
]))

stimmig = probe[1] == probe[3] == probe[5] == probe[6] == 0
BEFUNDE['Zeitzone belegt'] = (
    ('OK', 'vier abgeleitete Felder stimmen ohne Abweichung mit ' + ZONE + ' ueberein')
    if stimmig else
    ('ACHTUNG', 'die mitgelieferten Zeitfelder passen zu keiner der beiden Lesarten')
)

Eindeutig: Als Ortszeit gelesen stimmen alle vier Felder **ohne eine einzige
Abweichung**; als UTC gelesen weicht die Stunde in jeder Zeile ab. Der
Zeitstempel ist UTC, die Stadt rechnet in `America/Detroit` — und Silver tut
dasselbe. Das ist keine Annahme mehr, sondern gemessen.

Der Nebeneffekt betrifft das Ladefenster: Ein Jahr „2024" muss in Ortszeit
abgegrenzt werden. In UTC gezogen enthielte es Fälle vom 31.12.2023 und verlöre
welche vom Jahresende.

In [ ]:
display(con.execute(
    'SELECT trim(case_status) AS case_status, count(*) AS zeilen, '
    '       round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS prozent '
    'FROM d GROUP BY 1 ORDER BY zeilen DESC'
).df())

print('offense_category   :',
      con.execute('SELECT count(DISTINCT trim(offense_category)) FROM d').fetchone()[0], 'Werte')
print('offense_description:',
      con.execute('SELECT count(DISTINCT trim(offense_description)) FROM d').fetchone()[0], 'Werte')
print('state_offense_code :',
      con.execute('SELECT count(DISTINCT trim(state_offense_code)) FROM d').fetchone()[0], 'Werte')

## Schritt 10 – Fachliche Codes gegen die Referenz

`state_offense_code` ist ein MICR-Code des Staates Michigan. Damit Detroit mit
Chicago vergleichbar wird, muss jeder davon auf NIBRS abgebildet sein — sonst
fällt das Delikt aus der Auswertung.

Geprüft wird gegen `config/micr_nibrs_mapping.csv`, und zwar zweistufig: Ist der
Code überhaupt bekannt, und führt er zu einem NIBRS-Code?

In [ ]:
con.execute("CREATE OR REPLACE VIEW micr AS SELECT * FROM read_csv('"
            + str(PROJECT_ROOT / 'config' / 'micr_nibrs_mapping.csv') + "')")
bekannt, mit_nibrs = con.execute(
    'SELECT count(*) FILTER (WHERE m.micr_code IS NOT NULL), '
    "       count(*) FILTER (WHERE m.nibrs_code IS NOT NULL AND m.nibrs_code <> '') "
    'FROM d LEFT JOIN micr m ON trim(d.state_offense_code) = m.micr_code'
).fetchone()
print('Code in der Referenz bekannt :', f'{bekannt:,} / {N:,} = {bekannt / N:.2%}')
print('davon mit NIBRS-Code         :', f'{mit_nibrs:,} / {N:,} = {mit_nibrs / N:.2%}')

display(con.execute(
    "SELECT coalesce(m.match, '(unbekannt)') AS herkunft_der_zuordnung, "
    '       count(*) AS zeilen, '
    '       round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS prozent '
    'FROM d LEFT JOIN micr m ON trim(d.state_offense_code) = m.micr_code '
    'GROUP BY 1 ORDER BY zeilen DESC'
).df())

offen = con.execute(
    'SELECT trim(d.state_offense_code) AS code, '
    '       any_value(trim(d.offense_description)) AS delikt, count(*) AS zeilen '
    'FROM d LEFT JOIN micr m ON trim(d.state_offense_code) = m.micr_code '
    "WHERE m.nibrs_code IS NULL OR m.nibrs_code = '' "
    'GROUP BY 1 ORDER BY zeilen DESC'
).df()
if len(offen):
    display(offen)
BEFUNDE['MICR-Codes zugeordnet'] = (
    ('OK', f'{mit_nibrs:,} von {N:,} Zeilen ({mit_nibrs / N:.2%}) tragen einen NIBRS-Code')
    if mit_nibrs == N else
    ('ACHTUNG', f'{N - mit_nibrs:,} Zeilen ohne NIBRS-Code, {len(offen)} verschiedene Codes')
)

Alle Zeilen sind zugeordnet — aber die Spalte `match` zeigt, dass das nicht aus
einer einzigen Quelle kommt. 94,4 % stammen aus dem MICR-Handbuch selbst, der
Rest aus drei Ergänzungen: der Liste der Arrest Charges, einer Entscheidung
anhand von Detroits eigener File-Class-Angabe, und dem Anhang F von 2009 für
Codes, die das aktuelle Handbuch nicht mehr führt. Notebook 32 belegt jede
dieser Quellen.

Das ist ehrlicher, als eine einzige Zahl zu nennen: Die Zuordnung ist
vollständig, aber nicht überall gleich gut belegt.

## Schritt 11 – Zeitliche Abdeckung

Ein voller Jahrgang muss zwölf Monate haben und keinen Tag auslassen. Geprüft
wird in Ortszeit — nach Schritt 9 ist das die richtige Achse.

In [ ]:
lokal = "timezone('" + ZONE + "', epoch_ms(incident_occurred_at) AT TIME ZONE 'UTC')"
frueheste, spaeteste, ausserhalb = con.execute(
    'SELECT min(' + lokal + '), max(' + lokal + '), '
    '       count(*) FILTER (WHERE year(' + lokal + ') <> ' + str(JAHR) + ') FROM d'
).fetchone()
tage_ohne_fall = con.execute(
    'WITH vorhanden AS (SELECT DISTINCT CAST(' + lokal + ' AS DATE) AS tag FROM d), '
    '     alle AS (SELECT CAST(range AS DATE) AS tag FROM range('
    "         DATE '" + str(JAHR) + "-01-01', DATE '" + str(JAHR + 1) + "-01-01', INTERVAL 1 DAY)) "
    'SELECT count(*) FROM alle LEFT JOIN vorhanden USING (tag) WHERE vorhanden.tag IS NULL'
).fetchone()[0]
print('frueheste Tat  :', frueheste)
print('spaeteste Tat  :', spaeteste)
print('ausserhalb', JAHR, ':', ausserhalb)
print('Tage ohne Fall :', tage_ohne_fall)

display(con.execute(
    'SELECT month(' + lokal + ') AS monat, count(*) AS zeilen FROM d GROUP BY 1 ORDER BY 1'
).df())

BEFUNDE['Zeitliche Abdeckung'] = (
    ('OK', f'12 Monate, {tage_ohne_fall} Tage ohne Fall, nichts ausserhalb {JAHR}')
    if tage_ohne_fall == 0 and ausserhalb == 0 else
    ('ACHTUNG', f'{tage_ohne_fall} Tage ohne Fall, {ausserhalb} Zeilen ausserhalb {JAHR}')
)

Kein Tag ohne Fall, keine Zeile ausserhalb des Jahres. Der Sommer liegt etwa
25 % über dem Februar — ein bekanntes Muster und für eine der Hypothesen die
Ausgangslage, nicht ein Datenfehler.

## Schritt 12 – Rückwirkende Änderungen

Schritt 3 hat gezeigt, dass sich Fälle noch ändern. `updated_at` sagt, wie
verbreitet das ist: Wie lange nach der Tat wurde ein Fall zuletzt angefasst?

Die Antwort entscheidet, ob ein Delta-Verfahren für Detroit überhaupt tragen
würde — und sie ist der Grund, warum die Pipeline bei vollen Snapshots bleibt.

In [ ]:
gesamt, vorher, ueber_30, ueber_jahr, juengste = con.execute(
    'WITH t AS (SELECT epoch_ms(incident_occurred_at) AS tat, '
    '                  epoch_ms(updated_at) AS geaendert '
    '           FROM d WHERE updated_at IS NOT NULL) '
    'SELECT count(*), '
    '       count(*) FILTER (WHERE geaendert < tat), '
    '       count(*) FILTER (WHERE geaendert >= tat + INTERVAL 30 DAY), '
    '       count(*) FILTER (WHERE geaendert >= tat + INTERVAL 1 YEAR), '
    '       max(geaendert) FROM t'
).fetchone()
print('Zeilen mit updated_at         :', f'{gesamt:,}')
print('updated_at vor der Tat        :', vorher, ' (unmoeglich, aber selten)')
print('mehr als 30 Tage danach       :', f'{ueber_30:,} ({ueber_30 / gesamt:.1%})')
print('mehr als ein Jahr danach      :', f'{ueber_jahr:,} ({ueber_jahr / gesamt:.1%})')
print('juengste Aenderung im Bestand :', juengste)

BEFUNDE['Rueckwirkende Aenderungen'] = ('ACHTUNG', (
    f'{ueber_jahr / gesamt:.0%} der Faelle wurden mehr als ein Jahr nach der Tat '
    'noch angefasst - ein Delta nach Tatdatum wuerde sie verpassen'
))

Mehr als die Hälfte des Jahrgangs wurde über ein Jahr nach der Tat noch
geändert. Ein Delta-Verfahren, das nach **Tatdatum** nachlädt, würde diese
Korrekturen nie sehen. Detroit lädt deshalb den vollen Jahrgang neu — was bei
88'685 Zeilen und 97 Sekunden Laufzeit die einfachere und die richtigere
Variante ist.

Die drei Zeilen mit `updated_at` vor der Tat sind ein Fehler der Quelle. Zu
wenige, um etwas zu verschieben; zu eindeutig, um sie zu verschweigen.

## Schritt 13 – Schema-Drift

Die Ingestion pinnt 28 Felder. Was die API darüber hinaus liefert, landet
ungefiltert in `_extra` — ein neues Feld fällt so auf, statt still verloren zu
gehen. `_feature_json` hält zusätzlich die geparste Antwort je Zeile.

In [ ]:
mit_extra, ohne_json = con.execute(
    "SELECT count(*) FILTER (WHERE _extra IS NOT NULL AND _extra <> '{}'), "
    '       count(*) FILTER (WHERE _feature_json IS NULL) FROM roh'
).fetchone()
print('Zeilen mit _extra        :', mit_extra)
print('Zeilen ohne _feature_json:', ohne_json)
print('Manifest rows_with_extra :', manifest.get('rows_with_extra'))
print('schema_fingerprint       :', str(manifest.get('schema_fingerprint', ''))[:16] + '...')

BEFUNDE['Schema-Drift'] = (
    ('OK', 'kein unbekanntes Feld, Fingerabdruck des Schemas im Manifest festgehalten')
    if mit_extra == 0 else
    ('ACHTUNG', f'{mit_extra:,} Zeilen mit unbekannten Feldern in _extra')
)

## Schritt 14 – Die Befunde zusammen

In [ ]:
display(pd.DataFrame(
    [{'Pruefung': k, 'Status': s, 'Befund': b} for k, (s, b) in BEFUNDE.items()]
))

taten_ohne_geo = con.execute(
    'SELECT count(*) FROM (SELECT trim(case_id), trim(offense_description) FROM d '
    '                      GROUP BY 1, 2 HAVING count(*) FILTER (WHERE latitude IS NOT NULL) = 0)'
).fetchone()[0]
print('Zeilen (Anklagen)          :', f'{N:,}')
print('Taten                      :', f'{taten:,}')
print('davon raeumlich auswertbar :',
      f'{taten - taten_ohne_geo:,}',
      f'({(taten - taten_ohne_geo) / taten * 100:.2f} %)')

## Ergebnis und was daraus für Silver folgt

Der Jahrgang 2024 ist in gutem Zustand: vollständig geladen und nachgewiesen,
keine Lücke in der Zeitachse, kein unbekannter Deliktschlüssel, kein
Schema-Drift, `incident_entry_id` als tragfähiger Schlüssel, weniger Geo-Lücken
als bei Chicago.

Fünf Punkte muss die Silver-Schicht aber ausdrücklich behandeln:

| Befund | Was in Silver zu tun ist |
| --- | --- |
| Eine Zeile ist eine **Anklage**, nicht ein Delikt | Je Tat (`case_id` × `offense_description`) genau eine Zeile als zählend markieren; Gold nimmt nur diese |
| Textfelder sind unvorhersehbar mit Leerzeichen gepolstert | Jedes Textfeld beim Einlesen trimmen — sonst verdoppeln sich 81 Delikttypen auf 152 |
| `ESRI_OID` ist **kein** Schlüssel | Über `incident_entry_id` verknüpfen; `snapshot_ids_sha256` nur laufintern verwenden |
| Die Quelle korrigiert rückwirkend, mit neuem `updated_at` | Nach `incident_entry_id` deduplizieren, je Fall die Fassung mit dem jüngsten `updated_at` |
| `incident_occurred_at` ist UTC, die Stadt rechnet in Ortszeit | Beides führen: `occurred_at_utc` für den Vergleich, `occurred_at_local` für Tages- und Jahresgrenzen |

Dazu die kleineren Beobachtungen: drei Zeilen mit `updated_at` vor der Tat, ein
Viertel ohne `updated_in_ibr_at`. Beides betrifft zu wenige Fälle oder zu
nebensächliche Felder, um Ergebnisse zu verschieben, sollte aber bekannt sein.

**Was dieses Notebook über Chicago hinaus zeigt:** Ein abgeschlossenes Jahr ist
keine Konstante. Zwischen zwei Läufen im Abstand von gut 20 Stunden
wurden zehn Delikte umklassiert. Jede Zahl dieses Projekts gilt für einen Stand — deshalb
steht in jedem Manifest, aus welchen Läufen sie stammt, und deshalb hält
`docs/AENDERUNGEN.md` fest, wann sie sich geändert hat.

**Nächster Schritt:** der räumliche Join gegen die TIGER-Tracts. Dieses Notebook
liefert dafür die Erwartung — wie viele Taten überhaupt zuordenbar sein können,
und dass die Lücke hier nicht systematisch nach Delikttyp verzerrt ist.